# Ứng dụng 1 — Dự đoán bệnh tiểu đường

**Assignment 02 — Phát triển các Hệ thống Thông minh**
Nguyễn Duy Nghĩa · B23DCCN600 · D23CTPM01 · GVHD: PGS.TS Trần Đình Quế

Notebook này đi đúng chuỗi mà đề bài yêu cầu:

`Dữ liệu thô → Tìm hiểu → Làm sạch → Biểu diễn → Học → Đánh giá → Lưu trữ → Triển khai`

## 1. Định nghĩa bài toán

**Mục tiêu.** Dự đoán một bệnh nhân nữ gốc Pima có thuộc lớp dương tính tiểu đường
hay không, từ các chỉ số lâm sàng thu được trong một lần khám.

- $X$ = đặc trưng bệnh nhân (8 chỉ số lâm sàng và nhân trắc học)
- $y$ = lớp tiểu đường, $y \in \{0, 1\}$ — $1$ nghĩa là dương tính

Đây là bài toán **phân loại nhị phân có giám sát**.

**Vì sao bài toán này đáng làm.** Tiểu đường type 2 tiến triển âm thầm nhiều năm
trước khi có triệu chứng. Một mô hình sàng lọc chạy trên các chỉ số xét nghiệm
thường quy có thể chỉ ra ai cần làm nghiệm pháp dung nạp glucose chuyên sâu —
nghĩa là mô hình không thay thế chẩn đoán, nó **phân bổ nguồn lực chẩn đoán**.
Nhận định này quyết định luôn việc chọn độ đo ở mục 19: bỏ sót một ca bệnh
(FN) đắt hơn nhiều so với gọi nhầm một người khoẻ (FP).

## 2. Nguồn dữ liệu

| Mục | Giá trị |
|---|---|
| Tên tập dữ liệu | Pima Indians Diabetes Database |
| Nguồn Kaggle | https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database |
| Nguồn gốc | National Institute of Diabetes and Digestive and Kidney Diseases (UCI) |
| Số quan sát | 768 |
| Số thuộc tính | 9 (8 đặc trưng + 1 biến mục tiêu) |
| Biến mục tiêu | `Outcome` (0 = âm tính, 1 = dương tính) |
| Tệp cục bộ | `data/diabetes.csv` |

**Một quan sát là gì.** Mỗi dòng là **một lần khám của một bệnh nhân nữ, từ 21
tuổi trở lên, thuộc cộng đồng người Pima ở bang Arizona (Hoa Kỳ)**. Ràng buộc
nhân khẩu học này là một giới hạn thật của mô hình chứ không phải chi tiết vụn
vặt: cộng đồng Pima có tỷ lệ mắc tiểu đường type 2 cao bậc nhất thế giới, nên
ngưỡng quyết định học được ở đây **không thể suy rộng thẳng cho quần thể khác**.

In [1]:
# --- 3. Nạp dữ liệu ---
import json
import random
import warnings
from pathlib import Path

import joblib
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
matplotlib.use("Agg")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.bbox"] = "tight"
plt.rcParams["font.family"] = "DejaVu Sans"

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
DATA = ROOT / "data" / "diabetes.csv"
MODEL_DIR = ROOT / "model"
FIG_DIR = ROOT.parent / "report" / "figures"
MODEL_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA)
print("Đã nạp:", DATA)
print("Kích thước (N, cột):", df.shape)
df.head()

Đã nạp: D:\Python\HTTM_Assignment02\Assignment-02-Intelligent-System\diabetes\data\diabetes.csv
Kích thước (N, cột): (768, 9)


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


## 4. Khảo sát cấu trúc dữ liệu

Sáu lệnh khảo sát mà đề bài yêu cầu: `shape`, `head`, `info`, `describe`,
`isna().sum()`, `duplicated().sum()`.

In [2]:
print("--- df.shape ---"); print(df.shape)
print("\n--- df.info() ---"); df.info()
print("\n--- df.describe() ---")
display(df.describe().T.round(3))

--- df.shape ---
(768, 9)

--- df.info() ---
<class 'pandas.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB

--- df.describe() ---


,count,mean,std,min,25%,50%,75%,max
Pregnancies,768.0,3.845,3.370,0.000,1.000,3.000,6.000,17.00
Glucose,768.0,120.895,31.973,0.000,99.000,117.000,140.250,199.00
BloodPressure,768.0,69.105,19.356,0.000,62.000,72.000,80.000,122.00
SkinThickness,768.0,20.536,15.952,0.000,0.000,23.000,32.000,99.00
Insulin,768.0,79.799,115.244,0.000,0.000,30.500,127.250,846.00
BMI,768.0,31.993,7.884,0.000,27.300,32.000,36.600,67.10
DiabetesPedigreeFunction,768.0,0.472,0.331,0.078,0.244,0.372,0.626,2.42
Age,768.0,33.241,11.760,21.000,24.000,29.000,41.000,81.00
Outcome,768.0,0.349,0.477,0.000,0.000,0.000,1.000,1.00


In [3]:
print("--- Giá trị thiếu (NaN) theo cột ---")
print(df.isna().sum())
print("\n--- Số bản ghi trùng lặp hoàn toàn ---")
print(int(df.duplicated().sum()))
print("\n--- Phân bố biến mục tiêu ---")
print(df["Outcome"].value_counts())
print(df["Outcome"].value_counts(normalize=True).round(4))

--- Giá trị thiếu (NaN) theo cột ---
Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

--- Số bản ghi trùng lặp hoàn toàn ---
0

--- Phân bố biến mục tiêu ---
Outcome
0    500
1    268
Name: count, dtype: int64
Outcome
0    0.651
1    0.349
Name: proportion, dtype: float64


### Phân loại cột theo vai trò

| Nhóm | Cột |
|---|---|
| **Số (numerical)** | `Pregnancies`, `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, `BMI`, `DiabetesPedigreeFunction`, `Age` |
| **Phân loại (categorical)** | *(không có)* |
| **Văn bản (text)** | *(không có)* |
| **Mục tiêu (target)** | `Outcome` |

Toàn bộ 8 đặc trưng đều là số, nên **không cần mã hoá one-hot ở ứng dụng này**.
Đây là điểm khác biệt lớn so với Ứng dụng 2 (giá nhà, có biến phân loại) và
Ứng dụng 3 (thương mại điện tử, có văn bản) — sự khác biệt ấy chính là thứ
Chương X sẽ đem ra so sánh.

## 5. Phân tích chất lượng dữ liệu

`isna().sum()` trả về 0 ở mọi cột. **Kết luận "dữ liệu không thiếu" ở đây là sai.**

Tập Pima mã hoá giá trị thiếu bằng số **0**, không bằng `NaN`. Về mặt sinh lý học,
năm cột dưới đây **không thể bằng 0 trên một người còn sống**:

| Cột | Ý nghĩa | 0 có hợp lệ không? |
|---|---|---|
| `Glucose` | Nồng độ glucose huyết tương | Không — đường huyết 0 mg/dL là tử vong |
| `BloodPressure` | Huyết áp tâm trương | Không |
| `SkinThickness` | Độ dày nếp gấp da cơ tam đầu | Không |
| `Insulin` | Insulin huyết thanh 2 giờ | Không |
| `BMI` | Chỉ số khối cơ thể | Không |

Ngược lại, `Pregnancies = 0` **là hợp lệ** (chưa từng mang thai) và `Age = 0`
không xuất hiện. Vì vậy chỉ năm cột trên được xử lý.

Đây là ví dụ điển hình của **giá trị không hợp lệ đội lốt giá trị hợp lệ**:
`describe()` vẫn tính trung bình bình thường, không hàm nào báo lỗi, và mô hình
vẫn huấn luyện xong — chỉ là học sai.

In [4]:
# --- 6/8. Phân tích giá trị thiếu ẩn dưới dạng số 0 ---
INVALID_ZERO = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

zero_report = pd.DataFrame({
    "so_luong_bang_0": [(df[c] == 0).sum() for c in INVALID_ZERO],
})
zero_report.index = INVALID_ZERO
zero_report["ty_le_%"] = (zero_report["so_luong_bang_0"] / len(df) * 100).round(2)
display(zero_report.sort_values("ty_le_%", ascending=False))

fig, ax = plt.subplots(figsize=(7, 3.6))
z = zero_report.sort_values("ty_le_%")
ax.barh(z.index, z["ty_le_%"], color="#c0392b")
ax.set_xlabel("Tỷ lệ giá trị thiếu ẩn (%)")
ax.set_title("Ứng dụng 1 — Giá trị thiếu bị mã hoá thành số 0")
for i, v in enumerate(z["ty_le_%"]):
    ax.text(v + 0.6, i, f"{v}%", va="center", fontsize=9)
fig.savefig(FIG_DIR / "dia_missing.png")
plt.show()

,so_luong_bang_0,ty_le_%
Insulin,374,48.70
SkinThickness,227,29.56
BloodPressure,35,4.56
BMI,11,1.43
Glucose,5,0.65


**Quan sát.** `Insulin` thiếu 48,7% và `SkinThickness` thiếu 29,6% — hai cột này
gần như một nửa là dữ liệu bịa. `Glucose` và `BMI` chỉ thiếu dưới 1,5%.

**Diễn giải.** Insulin huyết thanh 2 giờ đòi hỏi nghiệm pháp dung nạp glucose
đường uống kéo dài — tốn thời gian và chi phí, nên nhiều bệnh nhân không được làm.
Đây là dạng thiếu **có hệ thống**, không phải ngẫu nhiên.

**Ý nghĩa với học máy.** Không được xoá dòng: xoá theo `Insulin` sẽ mất gần một
nửa tập dữ liệu vốn đã chỉ có 768 dòng. Cũng không được điền bằng trung bình
tính trên **toàn bộ** dữ liệu — đó là rò rỉ dữ liệu, vì trung bình ấy đã nhìn
thấy tập test. Cách xử lý đúng nằm ở mục 15: đưa việc điền khuyết **vào bên
trong pipeline**, để nó chỉ học thống kê từ tập train.

In [5]:
# --- 7. Phân tích trùng lặp ---
print("Số dòng trùng lặp hoàn toàn:", int(df.duplicated().sum()))
print("Số dòng trùng trên 8 đặc trưng (bỏ qua nhãn):",
      int(df.drop(columns=["Outcome"]).duplicated().sum()))

Số dòng trùng lặp hoàn toàn: 0
Số dòng trùng trên 8 đặc trưng (bỏ qua nhãn): 0


**Không có bản ghi trùng lặp.** Không cần thao tác khử trùng.

Nếu có, ta vẫn phải khử **trước khi chia train/test** — nếu không, một bản ghi
có thể nằm cả ở hai tập, khiến mô hình được chấm điểm trên chính dòng nó đã học,
và điểm test bị thổi phồng một cách không nhìn thấy được.

In [6]:
# --- 9. Phân tích ngoại lệ (outlier) bằng quy tắc IQR ---
NUM_COLS = [c for c in df.columns if c != "Outcome"]

rows = []
for c in NUM_COLS:
    s = df[c].replace(0, np.nan) if c in INVALID_ZERO else df[c]
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = int(((s < lo) | (s > hi)).sum())
    rows.append({"cot": c, "Q1": round(q1, 2), "Q3": round(q3, 2),
                 "nguong_duoi": round(lo, 2), "nguong_tren": round(hi, 2),
                 "so_ngoai_le": n_out,
                 "ty_le_%": round(n_out / s.notna().sum() * 100, 2)})
outlier_tbl = pd.DataFrame(rows).sort_values("ty_le_%", ascending=False)
display(outlier_tbl)

fig, axes = plt.subplots(2, 4, figsize=(15, 6.5))
for ax, c in zip(axes.ravel(), NUM_COLS):
    s = df[c].replace(0, np.nan) if c in INVALID_ZERO else df[c]
    sns.boxplot(y=s, ax=ax, color="#2980b9", width=0.45)
    ax.set_title(c, fontsize=10)
    ax.set_ylabel("")
fig.suptitle("Ứng dụng 1 — Hộp râu 8 đặc trưng (đã loại giá trị 0 không hợp lệ)", y=1.01)
fig.tight_layout()
fig.savefig(FIG_DIR / "dia_boxplot.png")
plt.show()

,cot,Q1,Q3,nguong_duoi,nguong_tren,so_ngoai_le,ty_le_%
4,Insulin,76.25,190.00,-94.38,360.62,24,6.09
6,DiabetesPedigreeFunction,0.24,0.63,-0.33,1.20,29,3.78
2,BloodPressure,64.00,80.00,40.00,104.00,14,1.91
7,Age,24.00,41.00,-1.50,66.50,9,1.17
5,BMI,27.50,36.60,13.85,50.25,8,1.06
3,SkinThickness,22.00,36.00,1.00,57.00,3,0.55
0,Pregnancies,1.00,6.00,-6.50,13.50,4,0.52
1,Glucose,99.00,141.00,36.00,204.00,0,0.00


**Quan sát.** `Insulin` và `DiabetesPedigreeFunction` có tỷ lệ ngoại lệ cao nhất;
phần lớn nằm ở đuôi phải.

**Diễn giải.** Insulin huyết thanh có phân phối lệch phải rất mạnh trong sinh lý
người: người kháng insulin nặng có thể cao gấp nhiều lần trung vị. `DiabetesPedigreeFunction`
là điểm số di truyền tổng hợp, cũng lệch phải theo thiết kế.

**Ý nghĩa với học máy.** Đây là **ngoại lệ thật, không phải lỗi nhập liệu** — chúng
mang đúng tín hiệu bệnh lý mà ta muốn mô hình học. Vì vậy **không cắt bỏ**. Thay vào
đó ta chọn hai biện pháp phòng thủ: (a) điền khuyết bằng **trung vị** thay vì trung
bình, vì trung vị bền với đuôi dài; (b) chuẩn hoá bằng `StandardScaler` để các mô
hình dựa trên khoảng cách (KNN, SVM) không bị một cột đuôi dài chi phối.

## 10. Phân tích khám phá dữ liệu (EDA)

In [7]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
cnt = df["Outcome"].value_counts().sort_index()
axes[0].bar(["Âm tính (0)", "Dương tính (1)"], cnt.values, color=["#27ae60", "#c0392b"])
for i, v in enumerate(cnt.values):
    axes[0].text(i, v + 8, f"{v}\n({v/len(df)*100:.1f}%)", ha="center", fontsize=10)
axes[0].set_title("Phân bố biến mục tiêu")
axes[0].set_ylabel("Số bệnh nhân")
axes[1].pie(cnt.values, labels=["Âm tính", "Dương tính"], autopct="%1.1f%%",
            colors=["#27ae60", "#c0392b"], startangle=90, explode=(0, 0.05))
axes[1].set_title("Tỷ lệ hai lớp")
fig.suptitle("Ứng dụng 1 — Mất cân bằng lớp", y=1.02)
fig.tight_layout()
fig.savefig(FIG_DIR / "dia_target.png")
plt.show()
print(cnt)
print("Tỷ lệ mất cân bằng:", round(cnt[0] / cnt[1], 2), ": 1")

Outcome
0    500
1    268
Name: count, dtype: int64
Tỷ lệ mất cân bằng: 1.87 : 1


**Quan sát.** 500 ca âm tính so với 268 ca dương tính — tỷ lệ khoảng 1,87 : 1
(65,1% / 34,9%).

**Diễn giải.** Mất cân bằng nhẹ. Không nghiêm trọng như bài toán phát hiện gian
lận (thường 1000 : 1), nhưng đủ để làm hỏng một độ đo.

**Ý nghĩa với học máy.** Một mô hình luôn đoán "âm tính" đạt **65,1% accuracy** mà
không học được gì. Đó chính là **baseline** ở mục 16, và mọi mô hình phải vượt qua
mốc ấy mới được coi là có giá trị. Hệ quả thứ hai: accuracy không đủ để xếp hạng
mô hình ở bài toán này — phải dùng thêm Recall và ROC-AUC.

In [8]:
fig, axes = plt.subplots(2, 4, figsize=(15, 7))
for ax, c in zip(axes.ravel(), NUM_COLS):
    s0 = df.loc[df.Outcome == 0, c].replace(0, np.nan) if c in INVALID_ZERO else df.loc[df.Outcome == 0, c]
    s1 = df.loc[df.Outcome == 1, c].replace(0, np.nan) if c in INVALID_ZERO else df.loc[df.Outcome == 1, c]
    sns.kdeplot(s0, ax=ax, fill=True, color="#27ae60", label="Âm tính", alpha=0.45)
    sns.kdeplot(s1, ax=ax, fill=True, color="#c0392b", label="Dương tính", alpha=0.45)
    ax.set_title(c, fontsize=10); ax.set_ylabel(""); ax.set_xlabel("")
axes.ravel()[0].legend(fontsize=8)
fig.suptitle("Ứng dụng 1 — Phân phối từng đặc trưng theo lớp", y=1.01)
fig.tight_layout()
fig.savefig(FIG_DIR / "dia_dist_by_class.png")
plt.show()

**Quan sát.** `Glucose` tách hai lớp rõ rệt nhất: đỉnh nhóm âm tính nằm quanh
110 mg/dL, nhóm dương tính quanh 140 mg/dL. `BMI` và `Age` tách vừa phải.
`BloodPressure` và `SkinThickness` gần như chồng lên nhau.

**Diễn giải.** Kết quả này khớp với y văn — glucose huyết tương lúc đói chính là
tiêu chuẩn chẩn đoán tiểu đường, nên nó phải là biến tách lớp mạnh nhất. Việc
mô hình "phát hiện lại" điều đã biết là một tín hiệu tốt: dữ liệu và pipeline
không bị hỏng ở đâu đó.

**Ý nghĩa với học máy.** `Glucose` sẽ chi phối mọi mô hình. Ngược lại,
`BloodPressure` và `SkinThickness` đóng góp rất ít — điều này sẽ được xác nhận
lại ở biểu đồ tầm quan trọng đặc trưng (mục 20) và là căn cứ để giao diện Web
chỉ hỏi 5 đặc trưng thay vì 8 (mục 23).

In [9]:
clean = df.copy()
clean[INVALID_ZERO] = clean[INVALID_ZERO].replace(0, np.nan)

fig, ax = plt.subplots(figsize=(8.5, 6.8))
corr = clean.corr(numeric_only=True)
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            square=True, linewidths=0.5, ax=ax, cbar_kws={"shrink": 0.8})
ax.set_title("Ứng dụng 1 — Ma trận tương quan Pearson")
fig.savefig(FIG_DIR / "dia_corr.png")
plt.show()

print("Tương quan với Outcome, sắp giảm dần:")
print(corr["Outcome"].drop("Outcome").sort_values(ascending=False).round(3))

Tương quan với Outcome, sắp giảm dần:
Glucose                     0.495
BMI                         0.314
Insulin                     0.303
SkinThickness               0.259
Age                         0.238
Pregnancies                 0.222
DiabetesPedigreeFunction    0.174
BloodPressure               0.171
Name: Outcome, dtype: float64


**Quan sát.** Tương quan với `Outcome` mạnh nhất là `Glucose` ($\approx 0{,}49$),
sau đó `BMI` ($\approx 0{,}31$) và `Age` ($\approx 0{,}24$). Giữa các đặc trưng,
cặp mạnh nhất là `Age`–`Pregnancies` ($\approx 0{,}54$) và
`SkinThickness`–`BMI` ($\approx 0{,}54$).

**Diễn giải.** Không có cặp nào vượt $0{,}8$, tức **không có đa cộng tuyến nghiêm
trọng**. Cặp `Age`–`Pregnancies` cao là hiển nhiên về mặt nhân khẩu học; cặp
`SkinThickness`–`BMI` cao vì cả hai đều đo lượng mỡ cơ thể.

**Ý nghĩa với học máy.** Hồi quy Logistic dùng được mà không cần loại cột — hệ số
vẫn diễn giải được. Nếu có cặp $>0{,}9$ thì hệ số hồi quy sẽ mất ổn định và ta đã
phải bỏ bớt một cột.

## 11. Kiểu đặc trưng

Cả 8 đặc trưng đều là **số liên tục hoặc số đếm**. Không có cột phân loại, không
có cột văn bản. Vì vậy bước "mã hoá" trong pipeline chuẩn là **rỗng** ở ứng dụng
này — toàn bộ công sức tiền xử lý dồn vào **điền khuyết** và **chuẩn hoá thang đo**.

## 12. Biểu diễn dữ liệu

Đây là mục trung tâm nối với Bài giảng 02. Chuỗi biến đổi:

$$\text{CSV} \rightarrow \text{DataFrame} \rightarrow \text{ma trận đặc trưng sạch}
\rightarrow \text{ma trận đã điền khuyết và chuẩn hoá} \rightarrow \text{đầu vào mô hình}$$

Một bệnh nhân trở thành một **vectơ đặc trưng**:

$$x_i = [x_{i1}, x_{i2}, \dots, x_{id}]^{T} \in \mathbb{R}^{d}$$

Toàn bộ tập dữ liệu trở thành **ma trận đặc trưng**:

$$X = \begin{bmatrix} x_1^{T} \\ x_2^{T} \\ \vdots \\ x_N^{T} \end{bmatrix} \in \mathbb{R}^{N \times d}$$

trong đó $N$ là số bệnh nhân và $d$ là số đặc trưng. Ô dưới đây in ra **một bản
ghi CSV gốc**, **vectơ đặc trưng tương ứng**, và **hình dạng ma trận** — đúng ba
thứ mà đề bài bắt buộc phải trình bày.

In [10]:
FEATURES = ["Glucose", "BMI", "Age", "Pregnancies", "DiabetesPedigreeFunction"]

print("=" * 74)
print("BƯỚC 1 — MỘT BẢN GHI CSV GỐC (dòng đầu tiên của tệp)")
print("=" * 74)
with open(DATA, encoding="utf-8") as f:
    print("Dòng tiêu đề :", f.readline().strip())
    print("Dòng dữ liệu :", f.readline().strip())

print("\n" + "=" * 74)
print("BƯỚC 2 — DÒNG ẤY SAU KHI ĐỌC VÀO DATAFRAME")
print("=" * 74)
display(df.head(1))

print("=" * 74)
print("BƯỚC 3 — VECTƠ ĐẶC TRƯNG x_1 (5 đặc trưng được chọn cho triển khai)")
print("=" * 74)
x1 = df.loc[0, FEATURES].to_numpy(dtype=float)
for name, val in zip(FEATURES, x1):
    print(f"   {name:<26} = {val}")
print("\n   x_1 =", np.round(x1, 4), " ∈ R^5")
print("   y_1 =", int(df.loc[0, "Outcome"]), "(1 = dương tính)")

print("\n" + "=" * 74)
print("BƯỚC 4 — HÌNH DẠNG MA TRẬN ĐẶC TRƯNG")
print("=" * 74)
X_all = df[FEATURES].to_numpy(dtype=float)
y_all = df["Outcome"].to_numpy(dtype=int)
print(f"   DataFrame gốc      : {df.shape}          (768 dòng × 9 cột)")
print(f"   Ma trận đặc trưng X: {X_all.shape}          → N = {X_all.shape[0]}, d = {X_all.shape[1]}")
print(f"   Vectơ mục tiêu   y : {y_all.shape}")
print(f"   Kiểu dữ liệu       : {X_all.dtype}")

BƯỚC 1 — MỘT BẢN GHI CSV GỐC (dòng đầu tiên của tệp)
Dòng tiêu đề : Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
Dòng dữ liệu : 6,148,72,35,0,33.6,0.627,50,1

BƯỚC 2 — DÒNG ẤY SAU KHI ĐỌC VÀO DATAFRAME


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1


BƯỚC 3 — VECTƠ ĐẶC TRƯNG x_1 (5 đặc trưng được chọn cho triển khai)
   Glucose                    = 148.0
   BMI                        = 33.6
   Age                        = 50.0
   Pregnancies                = 6.0
   DiabetesPedigreeFunction   = 0.627

   x_1 = [148.     33.6    50.      6.      0.627]  ∈ R^5
   y_1 = 1 (1 = dương tính)

BƯỚC 4 — HÌNH DẠNG MA TRẬN ĐẶC TRƯNG
   DataFrame gốc      : (768, 9)          (768 dòng × 9 cột)
   Ma trận đặc trưng X: (768, 5)          → N = 768, d = 5
   Vectơ mục tiêu   y : (768,)
   Kiểu dữ liệu       : float64


### Vì sao chỉ chọn 5 trong 8 đặc trưng

`Insulin` thiếu 48,7% và `SkinThickness` thiếu 29,6% — điền khuyết cho gần một nửa
số dòng nghĩa là **bịa ra gần một nửa cột đó**. `BloodPressure` thì gần như không
tách được hai lớp (biểu đồ mục 10) trong khi vẫn thiếu 4,6%.

Ba cột này bị loại vì cùng một lý do thực tế: **chi phí thu thập cao, đóng góp
thông tin thấp**. Giao diện Web ở mục 23 nhờ đó chỉ hỏi 5 con số — người dùng
điền được trong 20 giây, thay vì phải có kết quả xét nghiệm insulin 2 giờ.

Mục 18 sẽ **đo** cái giá của quyết định này bằng cách huấn luyện song song cả
hai phương án 5 đặc trưng và 8 đặc trưng.

## 13. Kỹ thuật đặc trưng

Ứng dụng này **không tạo thêm đặc trưng phái sinh**. Lý do: 8 chỉ số lâm sàng
đã là các đại lượng có ý nghĩa y học độc lập, và với chỉ 768 quan sát, thêm đặc
trưng tổ hợp sẽ làm tăng nguy cơ quá khớp nhanh hơn là tăng tín hiệu.

Toàn bộ "kỹ thuật đặc trưng" ở đây quy về hai phép biến đổi trong pipeline
(mục 15): thay 0 không hợp lệ bằng `NaN`, rồi điền khuyết bằng trung vị của
tập train.

## 14. Chia tập Train / Validation / Test

Đề bài yêu cầu $D = D_{\text{train}} \cup D_{\text{validation}} \cup D_{\text{test}}$
với ba tập độc lập, tỷ lệ khuyến nghị 70 / 15 / 15.

**Vì sao phải phân tầng (stratify).** Tập chỉ có 268 ca dương tính. Chia ngẫu
nhiên thuần tuý có thể cho ra tập test chỉ 30% dương tính trong khi train có 37%
— khi ấy điểm test đo cả sự lệch phân bố lẫn chất lượng mô hình, không tách được
hai thứ. `stratify=y` giữ tỷ lệ hai lớp gần như đồng nhất ở cả ba tập.

**Vì sao tập test không được ảnh hưởng đến huấn luyện.** Test tồn tại để ước
lượng sai số trên **dữ liệu chưa từng thấy**. Nếu bộ điền khuyết hoặc bộ chuẩn
hoá được `fit` trên toàn bộ dữ liệu, thì trung vị và độ lệch chuẩn dùng lúc huấn
luyện **đã chứa thông tin từ test** — điểm test khi ấy lạc quan giả tạo và mô
hình sẽ kém hơn hẳn khi ra thực tế. Đây chính là **rò rỉ dữ liệu**, và cách
chống duy nhất đáng tin là đặt mọi phép biến đổi vào bên trong `Pipeline`
(mục 15), rồi chỉ gọi `fit` trên tập train.

In [11]:
from sklearn.model_selection import train_test_split

X = df[FEATURES].copy()
y = df["Outcome"].copy()

# Tách test trước (15%), sau đó tách validation từ phần còn lại.
X_tmp, X_test, y_tmp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=RANDOM_SEED, stratify=y)
X_train, X_val, y_train, y_val = train_test_split(
    X_tmp, y_tmp, test_size=0.1765, random_state=RANDOM_SEED, stratify=y_tmp)

split_tbl = pd.DataFrame({
    "Tập": ["Train", "Validation", "Test", "Tổng"],
    "Số mẫu": [len(X_train), len(X_val), len(X_test), len(X)],
    "Tỷ lệ %": [round(len(t) / len(X) * 100, 1) for t in (X_train, X_val, X_test)] + [100.0],
    "Tỷ lệ dương tính %": [round(t.mean() * 100, 2) for t in (y_train, y_val, y_test)] + [round(y.mean() * 100, 2)],
})
display(split_tbl)
print("Hình dạng X_train:", X_train.shape, "| X_val:", X_val.shape, "| X_test:", X_test.shape)

,Tập,Số mẫu,Tỷ lệ %,Tỷ lệ dương tính %
0,Train,536,69.8,34.89
1,Validation,116,15.1,35.34
2,Test,116,15.1,34.48
3,Tổng,768,100.0,34.90


Hình dạng X_train: (536, 5) | X_val: (116, 5) | X_test: (116, 5)


Tỷ lệ dương tính ở cả ba tập đều xấp xỉ 34,9% — phân tầng đã làm đúng việc.

## 15. Pipeline tiền xử lý

Pipeline gồm ba bước, đóng gói thành **một đối tượng duy nhất** để lưu và tái dùng:

1. `FunctionTransformer` — thay giá trị 0 không hợp lệ bằng `NaN`.
2. `SimpleImputer(strategy="median")` — điền khuyết bằng **trung vị** của tập train.
   Chọn trung vị chứ không phải trung bình vì các cột đều lệch phải (mục 9);
   trung bình sẽ bị đuôi dài kéo lệch.
3. `StandardScaler` — đưa mỗi cột về trung bình 0, độ lệch chuẩn 1.
   Bắt buộc với KNN và SVM: `Glucose` có thang 0–200 còn
   `DiabetesPedigreeFunction` có thang 0–2,4; không chuẩn hoá thì khoảng cách
   Euclid gần như chỉ còn phản ánh mỗi `Glucose`.

Gói cả ba vào `Pipeline` là biện pháp chống rò rỉ dữ liệu: khi gọi
`pipeline.fit(X_train)`, cả `SimpleImputer` lẫn `StandardScaler` **chỉ nhìn thấy
tập train**. Khi dự đoán, `pipeline.transform` áp lại đúng các tham số đã học.

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler

INVALID_ZERO_IN_USE = ["Glucose", "BMI"]   # 3 cột kia đã bị loại khỏi FEATURES


def convert_invalid_zero_to_nan(data):
    '''Thay giá trị 0 không hợp lệ về mặt sinh lý bằng NaN.

    Hàm đặt ở cấp module (không phải lambda) để joblib có thể tuần tự hoá
    pipeline, và để REST API nạp lại được artifact.
    '''
    cleaned = data.copy()
    cols = [c for c in INVALID_ZERO_IN_USE if c in cleaned.columns]
    cleaned[cols] = cleaned[cols].replace(0, np.nan)
    return cleaned


preprocessor = Pipeline([
    ("invalid_zero", FunctionTransformer(convert_invalid_zero_to_nan, validate=False)),
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

preprocessor.fit(X_train)          # CHỈ fit trên train
Xtr = preprocessor.transform(X_train)
Xva = preprocessor.transform(X_val)
Xte = preprocessor.transform(X_test)

print("Trung vị học được từ tập train:")
for name, med in zip(FEATURES, preprocessor.named_steps["impute"].statistics_):
    print(f"   {name:<26} = {med:.3f}")
print("\nHình dạng đầu vào mô hình:")
print("   Xtr:", Xtr.shape, "| Xva:", Xva.shape, "| Xte:", Xte.shape)
print("   dtype:", Xtr.dtype)
print("\nKiểm tra chuẩn hoá trên tập train — trung bình ≈ 0, độ lệch chuẩn ≈ 1:")
print("   mean:", np.round(Xtr.mean(axis=0), 6))
print("   std :", np.round(Xtr.std(axis=0), 6))

Trung vị học được từ tập train:
   Glucose                    = 118.000
   BMI                        = 32.000
   Age                        = 29.000
   Pregnancies                = 3.000
   DiabetesPedigreeFunction   = 0.378

Hình dạng đầu vào mô hình:
   Xtr: (536, 5) | Xva: (116, 5) | Xte: (116, 5)
   dtype: float64

Kiểm tra chuẩn hoá trên tập train — trung bình ≈ 0, độ lệch chuẩn ≈ 1:
   mean: [-0.  0.  0. -0. -0.]
   std : [1. 1. 1. 1. 1.]


In [13]:
print("=" * 74)
print("BIỂU DIỄN CUỐI CÙNG ĐƯA VÀO MÔ HÌNH — bệnh nhân đầu tiên của tập train")
print("=" * 74)
raw_row = X_train.iloc[[0]]
print("Đầu vào thô (đơn vị gốc):")
display(raw_row)
print("Sau pipeline (vectơ đã chuẩn hoá):")
print("   ", np.round(Xtr[0], 4))
print("\n   Ma trận đầu vào mô hình: X ∈ R^{%d × %d}, dtype = %s" % (*Xtr.shape, Xtr.dtype))

BIỂU DIỄN CUỐI CÙNG ĐƯA VÀO MÔ HÌNH — bệnh nhân đầu tiên của tập train
Đầu vào thô (đơn vị gốc):


,Glucose,BMI,Age,Pregnancies,DiabetesPedigreeFunction
177,129,67.1,26,0,0.319


Sau pipeline (vectơ đã chuẩn hoá):
    [ 0.2531  5.2211 -0.6381 -1.1335 -0.4616]

   Ma trận đầu vào mô hình: X ∈ R^{536 × 5}, dtype = float64


## 16. Mô hình cơ sở (baseline)

Trước khi huấn luyện bất cứ mô hình nào, phải biết **ngưỡng vô nghĩa** nằm ở đâu.
`DummyClassifier(strategy="most_frequent")` luôn đoán lớp đa số (âm tính).
Mô hình nào không vượt được nó thì không có giá trị, dù accuracy trông cao.

In [14]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score, recall_score

dummy = DummyClassifier(strategy="most_frequent", random_state=RANDOM_SEED)
dummy.fit(Xtr, y_train)
yp = dummy.predict(Xte)
print("BASELINE — luôn đoán 'âm tính'")
print(f"   Accuracy : {accuracy_score(y_test, yp):.4f}   ← mốc phải vượt")
print(f"   Recall   : {recall_score(y_test, yp, zero_division=0):.4f}   ← bỏ sót TOÀN BỘ ca bệnh")
print(f"   F1       : {f1_score(y_test, yp, zero_division=0):.4f}")
BASELINE_ACC = accuracy_score(y_test, yp)

BASELINE — luôn đoán 'âm tính'
   Accuracy : 0.6552   ← mốc phải vượt
   Recall   : 0.0000   ← bỏ sót TOÀN BỘ ca bệnh
   F1       : 0.0000


Baseline đạt accuracy khoảng **65%** nhưng **Recall = 0** — nó bỏ sót 100% bệnh
nhân tiểu đường. Con số này minh hoạ chính xác vì sao accuracy một mình là độ đo
gây hiểu lầm ở bài toán mất cân bằng.

## 17. Huấn luyện mô hình

Đề bài yêu cầu **so sánh năm mô hình** cho bài toán tiểu đường:

| Mô hình | Họ | Vì sao đưa vào |
|---|---|---|
| Logistic Regression | Tuyến tính | Chuẩn tham chiếu trong y học; hệ số diễn giải được thành tỷ số chênh |
| K-Nearest Neighbors | Dựa trên khoảng cách | Không tham số; kiểm chứng xem bệnh nhân giống nhau có cùng nhãn không |
| Decision Tree | Dạng cây | Sinh ra luật đọc được, bác sĩ kiểm tra được |
| Random Forest | Tập hợp cây | Giảm phương sai của cây đơn; cho tầm quan trọng đặc trưng |
| SVM (RBF) | Biên lớn | Bắt ranh giới phi tuyến; cần dữ liệu đã chuẩn hoá |

`class_weight="balanced"` được bật ở Logistic Regression, Decision Tree,
Random Forest và SVM: nó nhân trọng số lỗi của lớp thiểu số lên, đẩy mô hình
ưu tiên **không bỏ sót ca bệnh** — đúng với ưu tiên lâm sàng đã nêu ở mục 1.

In [15]:
import time

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

MODELS = {
    "logistic_regression": ("Logistic Regression",
        LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_SEED)),
    "knn": ("K-Nearest Neighbors",
        KNeighborsClassifier(n_neighbors=15, weights="distance")),
    "decision_tree": ("Decision Tree",
        DecisionTreeClassifier(max_depth=5, min_samples_leaf=12,
                               class_weight="balanced", random_state=RANDOM_SEED)),
    "random_forest": ("Random Forest",
        RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=4,
                               class_weight="balanced", n_jobs=-1, random_state=RANDOM_SEED)),
    "svm_rbf": ("SVM (RBF)",
        SVC(kernel="rbf", C=1.0, gamma="scale", probability=True,
            class_weight="balanced", random_state=RANDOM_SEED)),
}

trained, train_times = {}, {}
for key, (label, model) in MODELS.items():
    t0 = time.perf_counter()
    model.fit(Xtr, y_train)
    train_times[key] = time.perf_counter() - t0
    trained[key] = model
    print(f"✓ {label:<24} huấn luyện xong trong {train_times[key]:.3f}s")

✓ Logistic Regression      huấn luyện xong trong 0.008s
✓ K-Nearest Neighbors      huấn luyện xong trong 0.008s
✓ Decision Tree            huấn luyện xong trong 0.004s


✓ Random Forest            huấn luyện xong trong 0.777s
✓ SVM (RBF)                huấn luyện xong trong 0.071s


## 18. So sánh mô hình

Tất cả được chấm trên **cùng một tập validation** — tập test còn được giữ nguyên
cho mục 19 để phép đo cuối cùng vẫn trung thực.

In [16]:
from sklearn.metrics import (accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score)


def score(model, Xs, ys):
    pred = model.predict(Xs)
    proba = model.predict_proba(Xs)[:, 1] if hasattr(model, "predict_proba") else None
    return {
        "Accuracy": accuracy_score(ys, pred),
        "Precision": precision_score(ys, pred, zero_division=0),
        "Recall": recall_score(ys, pred, zero_division=0),
        "F1": f1_score(ys, pred, zero_division=0),
        "ROC-AUC": roc_auc_score(ys, proba) if proba is not None else np.nan,
    }


rows = []
for key, (label, _) in MODELS.items():
    s = score(trained[key], Xva, y_val)
    s["Mô hình"] = label
    s["Thời gian huấn luyện (s)"] = round(train_times[key], 4)
    rows.append(s)

val_tbl = pd.DataFrame(rows).set_index("Mô hình")[
    ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "Thời gian huấn luyện (s)"]].round(4)
print("KẾT QUẢ TRÊN TẬP VALIDATION")
display(val_tbl.sort_values("ROC-AUC", ascending=False))

KẾT QUẢ TRÊN TẬP VALIDATION


,Accuracy,Precision,Recall,F1,ROC-AUC,Thời gian huấn luyện (s)
Mô hình,,,,,,
Random Forest,0.7759,0.6415,0.8293,0.7234,0.8374,0.7769
Logistic Regression,0.7328,0.6042,0.7073,0.6517,0.8354,0.0076
K-Nearest Neighbors,0.7759,0.7273,0.5854,0.6486,0.8122,0.0079
SVM (RBF),0.7672,0.6400,0.7805,0.7033,0.8111,0.0709
Decision Tree,0.6466,0.5000,0.8049,0.6168,0.7985,0.0040


In [17]:
# Đo cái giá của việc chỉ dùng 5 đặc trưng thay vì 8.
FEATURES_8 = ["Pregnancies", "Glucose", "BloodPressure", "SkinThickness",
              "Insulin", "BMI", "DiabetesPedigreeFunction", "Age"]
INVALID_ZERO_8 = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]


def make_pipe_8():
    def zero_to_nan_8(data):
        c = data.copy()
        cols = [x for x in INVALID_ZERO_8 if x in c.columns]
        c[cols] = c[cols].replace(0, np.nan)
        return c
    return Pipeline([
        ("invalid_zero", FunctionTransformer(zero_to_nan_8, validate=False)),
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler()),
    ])


X8 = df[FEATURES_8]
X8_tmp, X8_test, _, _ = train_test_split(X8, y, test_size=0.15, random_state=RANDOM_SEED, stratify=y)
X8_train, X8_val, _, _ = train_test_split(X8_tmp, y_tmp, test_size=0.1765, random_state=RANDOM_SEED, stratify=y_tmp)

p8 = make_pipe_8().fit(X8_train)
X8tr, X8va = p8.transform(X8_train), p8.transform(X8_val)

cmp_rows = []
for key, (label, _) in MODELS.items():
    m8 = MODELS[key][1].__class__(**MODELS[key][1].get_params())
    m8.fit(X8tr, y_train)
    s5 = score(trained[key], Xva, y_val)
    s8 = score(m8, X8va, y_val)
    cmp_rows.append({"Mô hình": label,
                     "ROC-AUC (5 đặc trưng)": round(s5["ROC-AUC"], 4),
                     "ROC-AUC (8 đặc trưng)": round(s8["ROC-AUC"], 4),
                     "Chênh lệch": round(s8["ROC-AUC"] - s5["ROC-AUC"], 4)})
feat_cmp = pd.DataFrame(cmp_rows).set_index("Mô hình")
print("CÁI GIÁ CỦA VIỆC RÚT GỌN TỪ 8 XUỐNG 5 ĐẶC TRƯNG")
display(feat_cmp)
print(f"Chênh lệch ROC-AUC trung bình: {feat_cmp['Chênh lệch'].mean():+.4f}")

CÁI GIÁ CỦA VIỆC RÚT GỌN TỪ 8 XUỐNG 5 ĐẶC TRƯNG


,ROC-AUC (5 đặc trưng),ROC-AUC (8 đặc trưng),Chênh lệch
Mô hình,,,
Logistic Regression,0.8354,0.8361,0.0007
K-Nearest Neighbors,0.8122,0.7935,-0.0187
Decision Tree,0.7985,0.7979,-0.0007
Random Forest,0.8374,0.8237,-0.0137
SVM (RBF),0.8111,0.8172,0.0062


Chênh lệch ROC-AUC trung bình: -0.0052


Chênh lệch ROC-AUC trung bình giữa hai phương án rất nhỏ. Nghĩa là ba đặc trưng
bị loại (`Insulin`, `SkinThickness`, `BloodPressure`) **gần như không mang thêm
thông tin dự báo** — trong khi chúng chiếm phần lớn giá trị thiếu và đòi hỏi xét
nghiệm tốn kém. Quyết định rút gọn ở mục 12 vì vậy có bằng chứng, không phải
phỏng đoán.

## 19. Đánh giá trên tập test

**Độ đo nào quan trọng nhất cho bài toán này?** — **Recall của lớp dương tính.**

Bốn ô của ma trận nhầm lẫn có ý nghĩa lâm sàng rất khác nhau:

| Ô | Ý nghĩa lâm sàng | Cái giá |
|---|---|---|
| TP | Có bệnh, được cảnh báo | Đúng ý đồ hệ thống |
| TN | Không bệnh, không cảnh báo | Đúng |
| **FP** | Không bệnh, bị cảnh báo nhầm | Một lần xét nghiệm xác nhận thừa — phiền, rẻ |
| **FN** | **Có bệnh, bị bỏ sót** | **Bệnh tiến triển âm thầm nhiều năm — đắt, có khi không hồi phục** |

FN đắt hơn FP rất nhiều, nên ta tối ưu Recall, dùng ROC-AUC làm tiêu chí xếp
hạng tổng thể (không phụ thuộc ngưỡng), và **không** dùng accuracy làm tiêu chí chính.

In [18]:
rows = []
for key, (label, _) in MODELS.items():
    s = score(trained[key], Xte, y_test)
    s["Mô hình"] = label
    rows.append(s)
test_tbl = pd.DataFrame(rows).set_index("Mô hình")[
    ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]].round(4)
test_tbl = test_tbl.sort_values("ROC-AUC", ascending=False)
print("KẾT QUẢ TRÊN TẬP TEST — bảng so sánh cuối cùng")
display(test_tbl)
print(f"\nBaseline accuracy = {BASELINE_ACC:.4f}")
print("Số mô hình vượt baseline:", int((test_tbl['Accuracy'] > BASELINE_ACC).sum()), "/", len(test_tbl))

KẾT QUẢ TRÊN TẬP TEST — bảng so sánh cuối cùng


,Accuracy,Precision,Recall,F1,ROC-AUC
Mô hình,,,,,
Random Forest,0.7759,0.6458,0.775,0.7045,0.8526
Logistic Regression,0.7414,0.5962,0.775,0.6739,0.8359
SVM (RBF),0.7414,0.5962,0.775,0.6739,0.8118
K-Nearest Neighbors,0.7586,0.6667,0.600,0.6316,0.7985
Decision Tree,0.6810,0.5231,0.850,0.6476,0.7735



Baseline accuracy = 0.6552
Số mô hình vượt baseline: 5 / 5


In [19]:
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay, confusion_matrix

fig, axes = plt.subplots(1, 5, figsize=(20, 4))
for ax, (key, (label, _)) in zip(axes, MODELS.items()):
    cm = confusion_matrix(y_test, trained[key].predict(Xte))
    ConfusionMatrixDisplay(cm, display_labels=["Âm", "Dương"]).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(label, fontsize=10)
    ax.set_xlabel("Dự đoán"); ax.set_ylabel("Thực tế")
fig.suptitle("Ứng dụng 1 — Ma trận nhầm lẫn trên tập test", y=1.04)
fig.tight_layout()
fig.savefig(FIG_DIR / "dia_confusion.png")
plt.show()

fig, ax = plt.subplots(figsize=(7, 5.6))
for key, (label, _) in MODELS.items():
    RocCurveDisplay.from_estimator(trained[key], Xte, y_test, ax=ax, name=label)
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Đoán ngẫu nhiên")
ax.set_title("Ứng dụng 1 — Đường cong ROC trên tập test")
ax.set_xlabel("Tỷ lệ dương tính giả (FPR)"); ax.set_ylabel("Tỷ lệ dương tính thật (TPR)")
ax.legend(fontsize=8, loc="lower right")
fig.savefig(FIG_DIR / "dia_roc.png")
plt.show()

In [20]:
fig, ax = plt.subplots(figsize=(9.5, 4.6))
plot_tbl = test_tbl[["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]]
plot_tbl.plot(kind="bar", ax=ax, width=0.82, colormap="viridis")
ax.axhline(BASELINE_ACC, color="red", ls="--", lw=1.4,
           label=f"Baseline accuracy = {BASELINE_ACC:.3f}")
ax.set_ylim(0, 1.0); ax.set_ylabel("Giá trị"); ax.set_xlabel("")
ax.set_title("Ứng dụng 1 — So sánh 5 mô hình trên tập test")
ax.legend(fontsize=8, ncol=2, loc="lower right")
plt.xticks(rotation=18, ha="right")
fig.savefig(FIG_DIR / "dia_model_comparison.png")
plt.show()

### Diễn giải ma trận nhầm lẫn

Đề bài yêu cầu **diễn giải chứ không chỉ hiển thị** ma trận nhầm lẫn. Ô dưới đây
quy đổi bốn ô của mô hình tốt nhất sang ngôn ngữ lâm sàng.

In [21]:
best_key = test_tbl.index[0]
best_id = [k for k, (lab, _) in MODELS.items() if lab == best_key][0]
best_model = trained[best_id]

cm = confusion_matrix(y_test, best_model.predict(Xte))
tn, fp, fn, tp = cm.ravel()
print(f"Mô hình tốt nhất theo ROC-AUC: {best_key}\n")
print(f"  TP = {tp:>3}  bệnh nhân có bệnh, được cảnh báo đúng")
print(f"  TN = {tn:>3}  bệnh nhân khoẻ, không bị cảnh báo")
print(f"  FP = {fp:>3}  bệnh nhân khoẻ bị cảnh báo nhầm → chịu thêm 1 xét nghiệm xác nhận")
print(f"  FN = {fn:>3}  bệnh nhân CÓ BỆNH bị bỏ sót → đây là sai lầm tốn kém nhất")
print(f"\n  Recall = TP/(TP+FN) = {tp}/{tp+fn} = {tp/(tp+fn):.4f}")
print(f"    → hệ thống bắt được {tp/(tp+fn)*100:.1f}% số ca bệnh thật trong tập test")
print(f"  Precision = TP/(TP+FP) = {tp}/{tp+fp} = {tp/(tp+fp):.4f}")
print(f"    → trong 100 lần cảnh báo thì {tp/(tp+fp)*100:.0f} lần là báo động thật")

Mô hình tốt nhất theo ROC-AUC: Random Forest

  TP =  31  bệnh nhân có bệnh, được cảnh báo đúng
  TN =  59  bệnh nhân khoẻ, không bị cảnh báo
  FP =  17  bệnh nhân khoẻ bị cảnh báo nhầm → chịu thêm 1 xét nghiệm xác nhận
  FN =   9  bệnh nhân CÓ BỆNH bị bỏ sót → đây là sai lầm tốn kém nhất

  Recall = TP/(TP+FN) = 31/40 = 0.7750
    → hệ thống bắt được 77.5% số ca bệnh thật trong tập test
  Precision = TP/(TP+FP) = 31/48 = 0.6458
    → trong 100 lần cảnh báo thì 65 lần là báo động thật


## 20. Phân tích sai số

In [22]:
rf = trained["random_forest"]
imp = pd.Series(rf.feature_importances_, index=FEATURES).sort_values()
fig, ax = plt.subplots(figsize=(7.5, 3.6))
ax.barh(imp.index, imp.values, color="#16a085")
for i, v in enumerate(imp.values):
    ax.text(v + 0.006, i, f"{v:.3f}", va="center", fontsize=9)
ax.set_xlabel("Tầm quan trọng (Gini importance)")
ax.set_title("Ứng dụng 1 — Tầm quan trọng đặc trưng (Random Forest)")
fig.savefig(FIG_DIR / "dia_importance.png")
plt.show()
print(imp.sort_values(ascending=False).round(4))

Glucose                     0.4011
BMI                         0.2040
Age                         0.1903
DiabetesPedigreeFunction    0.1299
Pregnancies                 0.0747
dtype: float64


In [23]:
proba = best_model.predict_proba(Xte)[:, 1]
pred = best_model.predict(Xte)
err = pd.DataFrame(X_test.reset_index(drop=True))
err["y_that"] = y_test.to_numpy()
err["y_du_doan"] = pred
err["xac_suat"] = proba.round(3)

fn_rows = err[(err.y_that == 1) & (err.y_du_doan == 0)]
fp_rows = err[(err.y_that == 0) & (err.y_du_doan == 1)]
print(f"Số ca bỏ sót (FN): {len(fn_rows)} — xem 5 ca:")
display(fn_rows.head())
print(f"\nSố ca báo động nhầm (FP): {len(fp_rows)} — xem 5 ca:")
display(fp_rows.head())

print("\nGlucose trung bình theo nhóm:")
print(f"  Ca đúng dương tính (TP): {err[(err.y_that==1)&(err.y_du_doan==1)].Glucose.mean():.1f} mg/dL")
print(f"  Ca bỏ sót        (FN): {fn_rows.Glucose.mean():.1f} mg/dL")
print(f"  Ca đúng âm tính  (TN): {err[(err.y_that==0)&(err.y_du_doan==0)].Glucose.mean():.1f} mg/dL")

Số ca bỏ sót (FN): 9 — xem 5 ca:


,Glucose,BMI,Age,Pregnancies,DiabetesPedigreeFunction,y_that,y_du_doan,xac_suat
1,123,32.0,35,4,0.226,1,0,0.500
3,124,34.0,38,5,0.220,1,0,0.498
12,109,34.8,26,4,0.905,1,0,0.452
17,119,27.1,33,6,1.318,1,0,0.479
18,78,31.0,26,3,0.248,1,0,0.086



Số ca báo động nhầm (FP): 17 — xem 5 ca:


,Glucose,BMI,Age,Pregnancies,DiabetesPedigreeFunction,y_that,y_du_doan,xac_suat
6,162,27.7,54,10,0.182,0,1,0.867
10,137,32.0,39,7,0.391,0,1,0.613
27,154,46.1,27,6,0.571,0,1,0.791
28,118,44.5,26,4,0.904,0,1,0.530
31,146,38.2,29,2,0.329,0,1,0.736



Glucose trung bình theo nhóm:
  Ca đúng dương tính (TP): 149.1 mg/dL
  Ca bỏ sót        (FN): 117.2 mg/dL
  Ca đúng âm tính  (TN): 100.7 mg/dL


**Quan sát.** Các ca bị bỏ sót (FN) có `Glucose` trung bình **thấp hơn hẳn** các
ca được bắt đúng, và nằm gần vùng giá trị của nhóm âm tính.

**Diễn giải.** Đây là những bệnh nhân đã mắc bệnh nhưng chưa biểu hiện tăng đường
huyết rõ — giai đoạn sớm, hoặc đang được kiểm soát bằng chế độ ăn. Với chỉ 5 chỉ
số ở một thời điểm, **không mô hình nào phân biệt được họ với người khoẻ**: thông
tin cần thiết (HbA1c, tiền sử theo thời gian) không nằm trong dữ liệu.

**Ý nghĩa với học máy.** Đây là **giới hạn của biểu diễn dữ liệu, không phải của
thuật toán**. Đổi sang mô hình mạnh hơn sẽ không sửa được; chỉ có thêm đặc trưng
mới sửa được. Đúng luận điểm trung tâm của Assignment 02.

## 21. Lựa chọn mô hình

Năm tiêu chí mà đề bài yêu cầu cân nhắc:

| Tiêu chí | Nhận định |
|---|---|
| Hiệu năng dự báo | Xếp theo ROC-AUC trên tập test — bảng mục 19 |
| Khả năng diễn giải | Logistic Regression và Decision Tree đọc được; SVM RBF là hộp đen |
| Chi phí tính toán | Mọi mô hình đều huấn luyện dưới 1 giây với $N = 768$; suy luận không đáng kể |
| Độ bền | Random Forest bền nhất với ngoại lệ nhờ trung bình hoá nhiều cây |
| Ràng buộc triển khai | Cả năm đều tuần tự hoá được bằng joblib; kích thước nhỏ |

Ở quy mô dữ liệu này, chi phí tính toán không phân biệt được các phương án, nên
quyết định thực chất nằm ở **hiệu năng + độ bền**. Ô dưới chọn mô hình theo
ROC-AUC trên tập test và ghi lại lý do.

In [24]:
print("Xếp hạng theo ROC-AUC trên tập test:")
display(test_tbl[["ROC-AUC", "Recall", "F1", "Accuracy"]])
print(f"\n→ Mô hình được chọn để triển khai: {best_key}")
print(f"   ROC-AUC = {test_tbl.loc[best_key, 'ROC-AUC']:.4f}")
print(f"   Recall  = {test_tbl.loc[best_key, 'Recall']:.4f}")
print(f"   F1      = {test_tbl.loc[best_key, 'F1']:.4f}")

Xếp hạng theo ROC-AUC trên tập test:


,ROC-AUC,Recall,F1,Accuracy
Mô hình,,,,
Random Forest,0.8526,0.775,0.7045,0.7759
Logistic Regression,0.8359,0.775,0.6739,0.7414
SVM (RBF),0.8118,0.775,0.6739,0.7414
K-Nearest Neighbors,0.7985,0.600,0.6316,0.7586
Decision Tree,0.7735,0.850,0.6476,0.6810



→ Mô hình được chọn để triển khai: Random Forest
   ROC-AUC = 0.8526
   Recall  = 0.7750
   F1      = 0.7045


## 22. Lưu trữ mô hình

Đề bài yêu cầu lưu **bốn thứ**: tiền xử lý, phép biến đổi đặc trưng, mô hình đã
huấn luyện, và cấu hình mô hình.

Ta lưu `preprocessor` (đã chứa cả điền khuyết lẫn chuẩn hoá) thành một tệp, lưu
cả năm mô hình để giao diện Web cho phép người dùng đổi mô hình, và lưu một tệp
`metadata.json` ghi thứ tự cột, ngưỡng hợp lệ và điểm số.

**Vì sao thứ tự cột phải được lưu.** `preprocessor.transform` nhận một mảng
theo **vị trí**, không theo tên. Nếu REST API dựng DataFrame theo thứ tự khác lúc
huấn luyện, giá trị `BMI` sẽ được chuẩn hoá bằng trung bình và độ lệch chuẩn của
`Age`. **Không lỗi nào được ném ra** — mô hình vẫn trả về một con số, chỉ là con
số sai. `metadata.json` là thứ chặn lỗi ấy.

In [25]:
joblib.dump(preprocessor, MODEL_DIR / "preprocessor.joblib")
print("✓ preprocessor.joblib")
for key, (label, _) in MODELS.items():
    # compress=3: mô hình rừng cây không nén chiếm hàng chục MB — quá lớn để đưa
    # vào kho mã nguồn. Nén zlib giảm khoảng 4 lần, gần như không ảnh hưởng tốc độ nạp.
    joblib.dump(trained[key], MODEL_DIR / f"{key}.joblib", compress=3)
    print(f"✓ {key}.joblib  ({label})")

metadata = {
    "application": "diabetes",
    "task": "binary_classification",
    "random_seed": RANDOM_SEED,
    "feature_columns": FEATURES,
    "target": "Outcome",
    "class_labels": {"0": "Âm tính (nguy cơ thấp)", "1": "Dương tính (nguy cơ cao)"},
    "invalid_zero_columns": INVALID_ZERO_IN_USE,
    "n_samples": int(len(df)),
    "feature_matrix_shape": list(X_all.shape),
    "split": {"train": len(X_train), "validation": len(X_val), "test": len(X_test)},
    "baseline_accuracy": round(float(BASELINE_ACC), 4),
    "best_model": best_id,
    "best_model_label": best_key,
    "test_metrics": {
        [k for k, (lab, _) in MODELS.items() if lab == idx][0]: {
            m: round(float(test_tbl.loc[idx, m]), 4)
            for m in ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]}
        for idx in test_tbl.index},
    "validation_metrics": {
        [k for k, (lab, _) in MODELS.items() if lab == idx][0]: {
            m: round(float(val_tbl.loc[idx, m]), 4)
            for m in ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]}
        for idx in val_tbl.index},
    "feature_importance": {k: round(float(v), 4) for k, v in imp.sort_values(ascending=False).items()},
    "confusion_matrix_best": {"tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp)},
    "feature_count_ablation": feat_cmp.reset_index().to_dict("records"),
}
(MODEL_DIR / "metadata.json").write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding="utf-8")
print("✓ metadata.json")
print("\nCác tệp artifact đã lưu:")
for p in sorted(MODEL_DIR.iterdir()):
    print(f"   {p.name:<32} {p.stat().st_size/1024:>8.1f} KB")

✓ preprocessor.joblib
✓ logistic_regression.joblib  (Logistic Regression)
✓ knn.joblib  (K-Nearest Neighbors)
✓ decision_tree.joblib  (Decision Tree)


✓ random_forest.joblib  (Random Forest)
✓ svm_rbf.joblib  (SVM (RBF))
✓ metadata.json

Các tệp artifact đã lưu:
   decision_tree.joblib                  2.3 KB
   knn.joblib                           13.9 KB
   logistic_regression.joblib            0.6 KB
   metadata.json                         3.4 KB
   preprocessor.joblib                   1.9 KB
   random_forest.joblib                458.0 KB
   svm_rbf.joblib                        8.7 KB


## 23. Kiểm thử suy luận

Bước cuối cùng chứng minh rằng **artifact đã lưu tự nó chạy được**: nạp lại từ
đĩa như REST API sẽ làm, rồi dự đoán trên một bệnh nhân mới.

Cần chứng minh hai điều: (a) không `fit` lại bất cứ thứ gì trên dữ liệu người
dùng — chỉ gọi `transform`; (b) kết quả nạp-lại **trùng khớp** với mô hình đang
nằm trong bộ nhớ. Nếu (b) sai thì artifact đã hỏng và dịch vụ Web sẽ trả kết quả
khác với thực nghiệm.

In [26]:
loaded_pre = joblib.load(MODEL_DIR / "preprocessor.joblib")
loaded_model = joblib.load(MODEL_DIR / f"{best_id}.joblib")
meta = json.loads((MODEL_DIR / "metadata.json").read_text(encoding="utf-8"))

patients = [
    {"Glucose": 168, "BMI": 38.2, "Age": 52, "Pregnancies": 6, "DiabetesPedigreeFunction": 0.85},
    {"Glucose": 92,  "BMI": 22.4, "Age": 24, "Pregnancies": 0, "DiabetesPedigreeFunction": 0.19},
]

for i, p in enumerate(patients, 1):
    frame = pd.DataFrame([p], columns=meta["feature_columns"])   # đúng thứ tự đã lưu
    vec = loaded_pre.transform(frame)                            # transform, KHÔNG fit
    cls = int(loaded_model.predict(vec)[0])
    prob = float(loaded_model.predict_proba(vec)[0][cls])
    print(f"--- Bệnh nhân {i} ---")
    print("   Đầu vào thô  :", p)
    print("   Vectơ chuẩn hoá:", np.round(vec[0], 4))
    print(f"   Dự đoán      : lớp {cls} — {meta['class_labels'][str(cls)]}")
    print(f"   Độ tin cậy   : {prob*100:.2f}%\n")

# Kiểm chứng: artifact nạp lại phải khớp mô hình trong bộ nhớ.
assert np.array_equal(loaded_model.predict(loaded_pre.transform(X_test)),
                      best_model.predict(Xte)), "Artifact nạp lại KHÔNG khớp!"
print("✓ Artifact nạp lại cho kết quả trùng khớp hoàn toàn trên 116 mẫu test.")
print("✓ Sẵn sàng cho REST API (api/REST_API.py).")

--- Bệnh nhân 1 ---
   Đầu vào thô  : {'Glucose': 168, 'BMI': 38.2, 'Age': 52, 'Pregnancies': 6, 'DiabetesPedigreeFunction': 0.85}
   Vectơ chuẩn hoá: [1.5757 0.894  1.5624 0.6404 1.175 ]
   Dự đoán      : lớp 1 — Dương tính (nguy cơ cao)
   Độ tin cậy   : 88.47%

--- Bệnh nhân 2 ---
   Đầu vào thô  : {'Glucose': 92, 'BMI': 22.4, 'Age': 24, 'Pregnancies': 0, 'DiabetesPedigreeFunction': 0.19}
   Vectơ chuẩn hoá: [-1.0016 -1.4716 -0.8073 -1.1335 -0.8592]
   Dự đoán      : lớp 0 — Âm tính (nguy cơ thấp)
   Độ tin cậy   : 97.78%



✓ Artifact nạp lại cho kết quả trùng khớp hoàn toàn trên 116 mẫu test.
✓ Sẵn sàng cho REST API (api/REST_API.py).


## Tóm tắt Ứng dụng 1

| Hạng mục | Kết quả |
|---|---|
| Bài toán | Phân loại nhị phân |
| Dữ liệu | Pima Indians Diabetes — 768 × 9 |
| Vấn đề chất lượng chính | Giá trị thiếu mã hoá thành số 0 (Insulin 48,7%) |
| Biểu diễn | $X \in \mathbb{R}^{768 \times 5}$, float64, đã điền khuyết + chuẩn hoá |
| Số mô hình so sánh | 5 |
| Độ đo chính | Recall (bỏ sót ca bệnh đắt hơn báo động nhầm) |
| Triển khai | Flask REST API + Web + giao diện Mobile |

Đóng góp riêng của ứng dụng này vào bài học chung: **hàm kiểm tra dữ liệu thiếu
có thể báo "sạch" trong khi gần một nửa một cột là bịa**. Chất lượng dữ liệu là
việc của người đọc dữ liệu, không phải của `isna()`.